In [1]:
import requests
import json
import numpy as np
import pandas as pd
import time
import warnings
import random
import re

pd.set_option('display.max_rows', 5000)
pd.set_option('display.max_columns', 5000)

from config import api_key

In [18]:
year_start = 2011 # leave 2011-2023 out of the first run to test the New Data Request check
year_end = 2025 # should be the year after the last full year of census data

zip_codes = ['90001', '90002', '90003', '90004', '90005', '90006', '90007', '90008', '90010', '90011',
             '90012', '90013', '90014', '90015', '90016', '90017', '90018', '90019',  '90020', '90021', 
             '90022', '90023', '90024', '90025', '90026', '90027', '90028', '90029', '90031', '90032',
             '90033', '90034', '90035', '90036', '90037', '90038', '90039', '90040', '90041', '90042', 
             '90043', '90044', '90045', '90046', '90047', '90048', '90049', '90056', '90057', '90058',
             '90059', '90061', '90062', '90063', '90064', '90065', '90066', '90067', '90068', '90069', 
             '90071', '90073', '90077', '90089', '90094', '90095', '90210', '90211', '90212',
             '90222', '90230', '90232', '90240', '90241', '90245', '90247', '90248', '90249', '90250',
             '90254', '90255', '90262', '90263', '90265', '90266', '90272', '90275', '90277', '90278',
             '90280', '90290', '90291', '90292', '90293', '90301', '90302', '90304', '90305', '90401', 
             '90402', '90403', '90404', '90405', '90501', '90502', '90601', '90660', '90670', '90704',
             '90710', '90717', '90723', '90731', '90732', '90744', '90745', '90802', '90803', '90804', 
             '90805', '90806', '90807', '90810', '90813', '90814', '90815', '91010', '91016', '91030', 
             '91040', '91042', '91104', '91105', '91201', '91202', '91204', '91205', '91214', '91302', 
             '91303', '91304', '91306', '91307', '91311', '91316', '91321', '91324', '91325', '91326', 
             '91330', '91331', '91335', '91340', '91342', '91343', '91344', '91345', '91352', '91356', 
             '91364', '91367', '91401', '91402', '91403', '91405', '91406', '91411', '91423', '91436', 
             '91501', '91502', '91504', '91505', '91506', '91601', '91602', '91604', '91605', 
             '91606', '91607', '91608', '91702', '91706', '91711', '91723', '91741', '91744', '91750', 
                # leave some ZIP codes out of the initial run to test the New Data Requests check
             '91765', '91766', '91767', '91801', '91803', '93536', '93551']

# removed ZIP codes 90090, 91523 because they do not return any data from the API

zip_codes_df = pd.DataFrame(zip_codes, columns=['ZIPCode'])

In [25]:
field_names_by_year_df = pd.read_csv('Data/CensusDataFieldsToQuery.csv')

data_fields_2024 = field_names_by_year_df\
    .loc[~(field_names_by_year_df.TranslatedName.isna())]\
    .FieldName2024.unique().tolist()
data_fields_2024.remove('zip code tabulation area')

field_rename_dict = pd.Series(field_names_by_year_df['TranslatedName'].values,
                              index=field_names_by_year_df['FieldName2024'].values).to_dict()

# Define Functions for API Calls

In [4]:
table_type_extensions = {
    'detailed': '',
    'subject': '/subject',
    'profile': '/profile',
    'comparison': '/cprofile'
}     

# need to chunk the number of variables into sets of 49 (at most) fields due to the 
    # 50-variable limit to the API (one variable being the ZIP code name)
def chunk_columns(field_list):    
    if len(field_list) > 50:
        list_of_field_lists = []
        for i in range(0, int(np.floor(len(field_list)) / 50)):
            list_of_field_lists.append(field_list[i*50:(i+1)*50])

        list_of_field_lists.append(field_list[(i+1)*50:])
        return(list_of_field_lists)
    else:
        return([field_list])

In [5]:
def write_api_call_url(year, field_list, zip_code_list, table_type):
    url = f'https://api.census.gov/data/{year}/acs/acs5' +\
        table_type_extensions[table_type] + '?get=' +\
        ','.join(field_list) +\
        '&for=zip%20code%20tabulation%20area:' +\
        ','.join(zip_code_list) +\
        ('&in=state:06' if year < 2020 else '') +\
        f'&key={api_key}'
    return(url)

In [6]:
# declare some global variables for debugging API call errors
api_call_error_parameters = []
misc_error_parameters = []
error_parameters_indices_to_pop = []

def log_api_error(year, field_list, zip_code_list, table_type, api_response):
    if api_response.status_code == 404: # only log Data Not Found errors, as these are likely rate limit errors
        api_call_error_parameters.append({'Year':year, 'Fields':field_list, 
                                          'ZIPCodes':zip_code_list, 'TableType':table_type})
        print(f'API call error; see parameters at index {len(api_call_error_parameters)}')
    else:
        # any other error indicates some unexpected issues with API call format or external problems 
        misc_error_parameters.append({'Year':year, 'Fields':field_list, 
                                                  'ZIPCodes':zip_code_list, 'TableType':table_type})
        raise Exception('Unexpected error occurred during attempted API call')

In [7]:
def make_api_call(year, field_list, zip_code_list, table_type, is_rerun=False, rerun_index=-1):
   
    url = write_api_call_url(year, field_list, zip_code_list, table_type)
    response = requests.get(url)
    try:
        queried_data = pd.DataFrame(response.json()[1:], columns=response.json()[0])
    except:
        time.sleep(6 + random.randint(-5000, 5000)/10000)
        response = requests.get(url)
        try:
            queried_data = pd.DataFrame(response.json()[1:], columns=response.json()[0])
        except:
            if not is_rerun:
                log_api_error(year, field_list, zip_code_list, table_type, response)
            queried_data = pd.DataFrame({'zip code tabulation area': zip_code_list})
    if is_rerun:
        error_parameters_indices_to_pop.append(rerun_index)
    
    if 'state' in queried_data.columns.tolist():
        queried_data.drop(['state'], axis=1, inplace=True)

    time.sleep(3 + random.randint(-5000, 5000)/10000)
    return(queried_data)

In [8]:
# need to make separate calls for each table we're querying
## there could be more error handling checks here to handle invalid year, ZIP code, or data field requests
def compile_api_data(year, field_list_2024_names, zip_code_list):
    
    if year and len(field_list_2024_names) > 0 and len(zip_code_list) > 0:
        
        if year == 2024:
            field_list = field_list_2024_names
        else:
            field_name_df = field_names_by_year_df[(
                field_names_by_year_df.Year == year) &
                (field_names_by_year_df.FieldName2024.isin(field_list_2024_names))]
            field_list = field_name_df.Name.values.tolist()

        if len(field_list) != len(field_list_2024_names):
            raise ValueError('Invalid field name(s): ' +
                             ', '.join([field for field in field_list if field_list not in field_list_2024_names]) +
                            f'for {year}')
        
        detailed_table_fields = chunk_columns([col for col in field_list if col[0] in ('B', 'C') and col[1] != 'P'])
        subject_table_fields = chunk_columns([col for col in field_list if col[0] == 'S'])
        data_profile_fields = chunk_columns([col for col in field_list if col[0:2] == 'DP'])
        comparison_table_fields = chunk_columns([col for col in field_list if col[0:2] == 'CP'])
        
        compiled_data = pd.DataFrame({'Year':year, 'zip code tabulation area': zip_code_list})

        if len(detailed_table_fields[0]) > 0:
            for chunk in detailed_table_fields:
                compiled_data = compiled_data.merge(make_api_call(year, chunk, zip_code_list, 'detailed'), 
                                                    on= 'zip code tabulation area', how='left')
        if len(subject_table_fields[0]) > 0:
            for chunk in subject_table_fields:
                compiled_data = compiled_data.merge(make_api_call(year, chunk, zip_code_list, 'subject'), 
                                                    on= 'zip code tabulation area', how='left')
        if len(data_profile_fields[0]) > 0:
            for chunk in data_profile_fields:
                compiled_data = compiled_data.merge(make_api_call(year, chunk, zip_code_list, 'profile'), 
                                                    on= 'zip code tabulation area', how='left')
        # ZIP code tabulation area not available for comparison profile tables
        
        if year != 2024:
            compiled_data = compiled_data.rename(columns=dict(field_name_df[['Name', 'FieldName2024']].values))
    else:
        raise Exception("Invalid data request") 

    return(compiled_data)

# Loading Any Saved Raw Data

In [26]:
try:
    saved_df = pd.read_csv('Data/LACensusRawAPIData.csv',
                          dtype={'zip code tabulation area': str})
    query_new_data = True
    new_data_df = saved_df.copy()
except:
    saved_df = compile_api_data(year_start, data_fields_2024, zip_codes)
    if year_end - year_start > 1:
        for i in range(year_start + 1, year_end):
            saved_df = pd.concat([saved_df, compile_api_data(i, data_fields_2024, zip_codes)])
    query_new_data = False

# Check for Any New Data Requests

In [27]:
missing_fields = [col for col in data_fields_2024 if col not in saved_df.columns] # query new columns for all years and ZIP codes
existing_fields = [col for col in data_fields_2024 if col not in missing_fields]

existing_rows = saved_df.groupby(['Year'])['zip code tabulation area'].apply(list)
existing_years = existing_rows.index
existing_zip_codes = saved_df['zip code tabulation area'].unique()
missing_years = [year for year in range(year_start, year_end) if year not in existing_years] 
    # query all data for all ZIP codes for missing years
missing_zip_codes = [zip_code for zip_code in zip_codes if zip_code not in existing_zip_codes] 
    # query all data for all existing years for missing ZIP codes
    
incomplete_data_list = []

for year in existing_years:
    yearly_df = saved_df.loc[saved_df.Year == year]
    yearly_existing_zip_codes = yearly_df['zip code tabulation area'].unique()
    yearly_complete_zip_codes = yearly_df.dropna()['zip code tabulation area'].unique()
    yearly_complete_fields = yearly_df.dropna(axis=1).columns
    
    incomplete_data_dict = {'Year': year, 
                            'Incomplete Fields': [col for col in existing_fields if col not in yearly_complete_fields], 
                            'Incomplete ZIP Codes': [zip_code for zip_code in yearly_existing_zip_codes if 
                                                     zip_code not in yearly_complete_zip_codes],
                            'Missing ZIP Codes': [zip_code for zip_code in existing_zip_codes if
                                                 zip_code not in yearly_existing_zip_codes]}
    
    
    if len(incomplete_data_dict['Incomplete Fields']) > 0 or \
        len(incomplete_data_dict['Incomplete ZIP Codes']) > 0 or \
        len(incomplete_data_dict['Missing ZIP Codes']) > 0 :
        incomplete_data_list.append(incomplete_data_dict)

In [28]:
print(missing_fields)
print(missing_years)
print(missing_zip_codes)
print(incomplete_data_list)

['B08105A_003E', 'B08105A_002E', 'B08105A_004E', 'B08105A_006E', 'B08105A_005E', 'B08105A_007E', 'C24010A_008E', 'C24010A_009E', 'C24010A_012E', 'C24010A_013E', 'C24010A_011E', 'C24010A_010E', 'C24010A_002E', 'C24010A_003E', 'C24010A_006E', 'C24010A_007E', 'C24010A_005E', 'C24010A_004E']
[]
[]
[]


# Extracting New Data from the API

## Making API calls for new columns

In [29]:
if query_new_data and len(missing_fields) > 0:
    first_year = existing_years[0]
    new_columns_df = compile_api_data(first_year, missing_fields, existing_rows[first_year])
    for year in existing_years[1:]:
        new_columns_df = pd.concat([new_columns_df, compile_api_data(year, missing_fields, existing_rows[year])])
    new_data_df = new_data_df.merge(new_columns_df, on=['Year', 'zip code tabulation area'], how='left')\
        [['Year', 'zip code tabulation area'] + data_fields_2024]

## Making API calls for missing years and ZIP codes

In [52]:
if query_new_data:
    if len(missing_years) > 0:
        for year in missing_years:
            new_data_df = pd.concat([new_data_df, compile_api_data(year, data_fields_2024, zip_codes)])
    if len(missing_zip_codes) > 0:
        zip_code_exists = True
        for year in existing_years:
            if zip_code_exists:
                try:
                    new_data_df = pd.concat([new_data_df, compile_api_data(year, data_fields_2024, missing_zip_codes)])
                except:
                    zip_code_exists = False
    new_data_df = new_data_df.sort_values(['Year', 'zip code tabulation area'])

## Making API calls to fill in any other missing data
This check is helpful for completing any API calls that failed in previous runs without being successfuly rerun

In [53]:
if query_new_data:
    if len(incomplete_data_list) > 0:
        for incomplete_data_dict in incomplete_data_list:
            year = incomplete_data_dict['Year']
            year_incomplete_fields = incomplete_data_dict['Incomplete Fields']
            year_incomplete_zip_codes = incomplete_data_dict['Incomplete ZIP Codes']
            year_missing_zip_codes = incomplete_data_dict['Missing ZIP Codes']
            
            if len(year_missing_zip_codes) > 0:
                new_data_df = pd.concat([new_data_df, 
                                            compile_api_data(year, missing_fields, year_missing_zip_codes)])
            if len(year_incomplete_fields) > 0 and len(year_incomplete_zip_codes) > 0:
                year_incomplete_df = new_data_df.loc[(new_data_df.Year == year) &
                    (new_data_df['zip code tabulation area'].isin(year_incomplete_zip_codes))]
                
                year_fill_incomplete_df = compile_api_data(year, year_incomplete_fields, year_incomplete_zip_codes)
                year_complete_df = year_incomplete_df[[col for col in year_incomplete_df.columns 
                                                       if col not in year_incomplete_fields]]\
                    .merge(year_fill_incomplete_df, on=['Year', 'zip code tabulation area'], how='left')
                new_data_df = pd.concat([new_data_df.loc[(new_data_df.Year != year) | 
                                                         ~(new_data_df['zip code tabulation area']\
                                                               .isin(year_incomplete_zip_codes))], year_complete_df])
            
    new_data_df = new_data_df.sort_values(['Year', 'zip code tabulation area'])

## Attempting to rerun API calls that returned errors

In [30]:
print(api_call_error_parameters)

[]


In [54]:
error_parameters_indices_to_pop = []

for i in range(len(api_call_error_parameters)):
    error_parameters_dict = api_call_error_parameters[i]
    error_year = error_parameters_dict['Year']
    error_fields = error_parameters_dict['Fields']
    error_zip_codes = error_parameters_dict['ZIPCodes']
    
    existing_data = new_data_df.loc[(new_data_df.Year == error_year) & 
                                   (new_data_df['zip code tabulation area']).isin(error_zip_codes)]\
        [['Year', 'zip code tabulation area'] + [col for col in data_fields_2024 if col not in error_fields]]
    
    if len(existing_data) == 0:
        existing_data = pd.DataFrame({'Year': error_year, 'zip code tabulation area': error_zip_codes})
    
    rerun_api_df = make_api_call(error_year, error_fields, error_zip_codes, 
                                 error_parameters_dict['TableType'], is_rerun=True)
    
    new_data_df = pd.concat([new_data_df.loc[(new_data_df.Year != error_year) | 
                                             ~(new_data_df['zip code tabulation area'].isin(error_zip_codes))], 
        existing_data.merge(rerun_api_df, by=['Year', 'zip code tabulation area'], how='left')])    

for parameter_index in error_parameters_indices_to_pop:
    api_call_error_parameters.pop(parameter_index)
    
if len(error_parameters_indices_to_pop) > 0:
    new_data_df = new_data_df.sort_values(['Year', 'zip code tabulation area'])

# Transforming and Loading the Data

## Loading the raw data

In [41]:
if new_data_df[['Year', 'zip code tabulation area'] + data_fields_2024].size >= saved_df.size:
    new_data_df[['Year', 'zip code tabulation area'] + data_fields_2024]\
        .drop_duplicates().sort_values(['Year', 'zip code tabulation area'])\
        .to_csv('Data/LACensusRawAPIData.csv', index=False)
else:
    raise Exception('Warning: Extracted dataset smaller than saved data')

## Transforming the raw data

In [42]:
transformed_data_df = pd.read_csv('Data/LACensusRawAPIData.csv', 
                                  dtype={'zip code tabulation area':str}).rename(columns=field_rename_dict)

transformed_data_df.head()

,Year,ZIPCode,FemaleCommutersBiking,FemaleCommutersCarpooling,FemaleCommutersDriving,FemaleCommutersPublicTransit,FemaleCommutersTaxiMotorcycleAndOther,FemaleCommutersWalking,FemaleCommutersWorkingFromHome,MaleCommutersBiking,MaleCommutersCarpooling,MaleCommutersDriving,MaleCommutersPublicTransit,MaleCommutersTaxiMotorcycleAndOther,MaleCommutersWalking,MaleCommutersWorkingFromHome,Age16to19CommutersCarpooling,Age20to24CommutersCarpooling,Age25to44CommutersCarpooling,Age45to54CommutersCarpooling,Age55to59CommutersCarpooling,Age60to64CommutersCarpooling,Age65AndOverCommutersCarpooling,Age16to19CommutersDriving,Age20to24CommutersDriving,Age25to44CommutersDriving,Age45to54CommutersDriving,Age55to59CommutersDriving,Age60to64CommutersDriving,Age65AndOverCommutersDriving,Age16to19CommutersPublicTransit,Age20to24CommutersPublicTransit,Age25to44CommutersPublicTransit,Age45to54CommutersPublicTransit,Age55to59CommutersPublicTransit,Age60to64CommutersPublicTransit,Age65AndOverCommutersPublicTransit,Age16to19CommutersTaxiBikeAndOther,Age20to24CommutersTaxiBikeAndOther,Age25to44CommutersTaxiBikeAndOther,Age45to54CommutersTaxiBikeAndOther,Age55to59CommutersTaxiBikeAndOther,Age60to64CommutersTaxiBikeAndOther,Age65AndOverCommutersTaxiBikeAndOther,Age16to19CommutersWalking,Age20to24CommutersWalking,Age25to44CommutersWalking,Age45to54CommutersWalking,Age55to59CommutersWalking,Age60to64CommutersWalking,Age65AndOverCommutersWalking,Age16to19CommutersWorkingFromHome,Age20to24CommutersWorkingFromHome,Age25to44CommutersWorkingFromHome,Age45to54CommutersWorkingFromHome,Age55to59CommutersWorkingFromHome,Age60to64CommutersWorkingFromHome,Age65AndOverCommutersWorkingFromHome,WhiteCommutersCarpooling,WhiteCommutersDriving,WhiteCommutersPublicTransit,WhiteCommutersTaxiBikeAndOther,WhiteCommutersWalking,WhiteCommutersWorkingFromHome,BlackCommutersCarpooling,BlackCommutersDriving,BlackCommutersPublicTransit,BlackCommutersTaxiBikeAndOther,BlackCommutersWalking,BlackCommutersWorkingFromHome,IndigenousCommutersCarpooling,IndigenousCommutersDriving,IndigenousCommutersPublicTransit,IndigenousCommutersTaxiBikeAndOther,IndigenousCommutersWalking,IndigenousCommutersWorkingFromHome,AsianCommutersCarpooling,AsianCommutersDriving,AsianCommutersPublicTransit,AsianCommutersTaxiBikeAndOther,AsianCommutersWalking,AsianCommutersWorkingFromHome,PacificIslanderCommutersCarpooling,PacificIslanderCommutersDriving,PacificIslanderCommutersPublicTransit,PacificIslanderCommutersTaxiBikeAndOther,PacificIslanderCommutersWalking,PacificIslanderCommutersWorkingFromHome,OtherRaceCommutersCarpooling,OtherRaceCommutersDriving,OtherRaceCommutersPublicTransit,OtherRaceCommutersTaxiBikeAndOther,OtherRaceCommutersWalking,OtherRaceCommutersWorkingFromHome,TwoOrMoreRacesCommutersCarpooling,TwoOrMoreRacesCommutersDriving,TwoOrMoreRacesCommutersPublicTransit,TwoOrMoreRacesCommutersTaxiBikeAndOther,TwoOrMoreRacesCommutersWalking,TwoOrMoreRacesCommutersWorkingFromHome,HispanicCommutersCarpooling,HispanicCommutersDriving,HispanicCommutersPublicTransit,HispanicCommutersTaxiBikeAndOther,HispanicCommutersWalking,HispanicCommutersWorkingFromHome,CommutersBiking,CommutersCarpooling,CommutersDriving,CommutersMotorcycle,CommutersOtherMeans,CommutersPublicTransit,CommutersTaxi,CommutersWalking,CommutersWorkingFromHome,WhiteFemaleWorkers,WhiteFemaleWorkersManagementBusinesScienceAndArts,WhiteFemaleWorkersResourcesConstructionAndMaintenance,WhiteFemaleWorkersProductionTransportAndMaterialMoving,WhiteFemaleWorkersSalesAndOffice,WhiteFemaleWorkersService,WhiteMaleWorkers,WhiteMaleWorkersManagementBusinesScienceAndArts,WhiteMaleWorkersResourcesConstructionAndMaintenance,WhiteMaleWorkersProductionTransportAndMaterialMoving,WhiteMaleWorkersSalesAndOffice,WhiteMaleWorkersService,BlackFemaleWorkers,BlackFemaleWorkersManagementBusinesScienceAndArts,BlackFemaleWorkersResourcesConstructionAndMaintenance,BlackFemaleWorkersProductionTransportAndMaterialMoving,BlackFemaleWorkersSalesAndOffice,BlackFemaleWorkersService,Bla

## Selecting rows to validate the data against the Census Bureau's tables

In [6]:
random_years = random.sample(transformed_data_df.Year.unique().tolist(), 2)
random_zip_codes = random.sample(transformed_data_df.ZIPCode.unique().tolist(), 5)

In [7]:
# [2011, 2019] # checked all fields for 5 random ZIP codes
# [2012, 2016] # checked all fields for 91331
# [2013, 2014, 2015, 2017, 2018, 2020, 2021, 2022, 2023] # checked DP05, B08301, and B08105I fields for 91331

In [8]:
transformed_data_df.loc[(transformed_data_df.Year == 2024) &
                       (transformed_data_df.ZIPCode == '91331')]

,Year,ZIPCode,MalePopulation,FemalePopulation,AgeUnder5,Age5to9,Age10to14,Age15to19,Age20to24,Age25to34,Age35to44,Age45to54,Age55to59,Age60to64,Age65to74,Age75to84,Age85AndOver,TwoOrMoreRacesPopulation,BlackPopulation,IndigenousPopulation,AsianPopulation,PacificIslanderPopulation,OtherRacePopulation,HispanicPopulation,WhiteNonHispanicPopulation,CommutersDriving,CommutersCarpooling,CommutersPublicTransit,CommutersTaxi,CommutersMotorcycle,CommutersBiking,CommutersWalking,CommutersOtherMeans,CommutersWorkingFromHome,MaleCommutersDriving,MaleCommutersCarpooling,MaleCommutersPublicTransit,MaleCommutersBiking,MaleCommutersWalking,MaleCommutersTaxiMotorcycleAndOther,MaleCommutersWorkingFromHome,FemaleCommutersDriving,FemaleCommutersCarpooling,FemaleCommutersPublicTransit,FemaleCommutersBiking,FemaleCommutersWalking,FemaleCommutersTaxiMotorcycleAndOther,FemaleCommutersWorkingFromHome,Age16to19CommutersDriving,Age20to24CommutersDriving,Age25to44CommutersDriving,Age45to54CommutersDriving,Age55to59CommutersDriving,Age60to64CommutersDriving,Age65AndOverCommutersDriving,Age16to19CommutersCarpooling,Age20to24CommutersCarpooling,Age25to44CommutersCarpooling,Age45to54CommutersCarpooling,Age55to59CommutersCarpooling,Age60to64CommutersCarpooling,Age65AndOverCommutersCarpooling,Age16to19CommutersPublicTransit,Age20to24CommutersPublicTransit,Age25to44CommutersPublicTransit,Age45to54CommutersPublicTransit,Age55to59CommutersPublicTransit,Age60to64CommutersPublicTransit,Age65AndOverCommutersPublicTransit,Age16to19CommutersWalking,Age20to24CommutersWalking,Age25to44CommutersWalking,Age45to54CommutersWalking,Age55to59CommutersWalking,Age60to64CommutersWalking,Age65AndOverCommutersWalking,Age16to19CommutersTaxiBikeAndOther,Age20to24CommutersTaxiBikeAndOther,Age25to44CommutersTaxiBikeAndOther,Age45to54CommutersTaxiBikeAndOther,Age55to59CommutersTaxiBikeAndOther,Age60to64CommutersTaxiBikeAndOther,Age65AndOverCommutersTaxiBikeAndOther,Age16to19CommutersWorkingFromHome,Age20to24CommutersWorkingFromHome,Age25to44CommutersWorkingFromHome,Age45to54CommutersWorkingFromHome,Age55to59CommutersWorkingFromHome,Age60to64CommutersWorkingFromHome,Age65AndOverCommutersWorkingFromHome,BlackCommutersDriving,BlackCommutersCarpooling,BlackCommutersPublicTransit,BlackCommutersWalking,BlackCommutersTaxiBikeAndOther,BlackCommutersWorkingFromHome,IndigenousCommutersDriving,IndigenousCommutersCarpooling,IndigenousCommutersPublicTransit,IndigenousCommutersWalking,IndigenousCommutersTaxiBikeAndOther,IndigenousCommutersWorkingFromHome,AsianCommutersDriving,AsianCommutersCarpooling,AsianCommutersPublicTransit,AsianCommutersWalking,AsianCommutersTaxiBikeAndOther,AsianCommutersWorkingFromHome,PacificIslanderCommutersDriving,PacificIslanderCommutersCarpooling,PacificIslanderCommutersPublicTransit,PacificIslanderCommutersWalking,PacificIslanderCommutersTaxiBikeAndOther,PacificIslanderCommutersWorkingFromHome,OtherRaceCommutersDriving,OtherRaceCommutersCarpooling,OtherRaceCommutersPublicTransit,OtherRaceCommutersWalking,OtherRaceCommutersTaxiBikeAndOther,OtherRaceCommutersWorkingFromHome,TwoOrMoreRacesCommutersDriving,TwoOrMoreRacesCommutersCarpooling,TwoOrMoreRacesCommutersPublicTransit,TwoOrMoreRacesCommutersWalking,TwoOrMoreRacesCommutersTaxiBikeAndOther,TwoOrMoreRacesCommutersWorkingFromHome,WhiteNonHispanicCommutersDriving,WhiteNonHispanicCommutersCarpooling,WhiteNonHispanicCommutersPublicTransit,WhiteNonHispanicCommutersWalking,WhiteNonHispanicCommutersTaxiBikeAndOther,WhiteNonHispanicCommutersWorkingFromHome,HispanicCommutersDriving,HispanicCommutersCarpooling,HispanicCommutersPublicTransit,HispanicCommutersWalking,HispanicCommutersTaxiBikeAndOther,HispanicCommutersWorkingFromHome
2685,2024,91331,50101,47507,5457,5477,7324,7541,7220,15233,12324,13814,5892,4728,7571,3925,1102,40886,1877,1131,4633,87,37444,86227,3745,33044,5473,1131,260,65,20,435,618,3003,19669,2810,549,20,239,500,1165,13375,2663,582,0,196,443,1838,522,3040,14877,8003,3032,1961,1609,162,783,2454,1067,514,288,205,2

By checking comprehensive data for five ZIP codes across two years, checking all fields again for ZIP code 91331 for two more years, and checking the fields from three of the tables for ZIP code 91331 for the rest of the years, we can be reasonably confident the pipeline is working as intended

## Performing minor transformations

In [44]:
transformed_data_df['CommutersNonCar'] = transformed_data_df.CommutersPublicTransit + \
    transformed_data_df.CommutersMotorcycle + transformed_data_df.CommutersBiking + \
    transformed_data_df.CommutersWalking + transformed_data_df.CommutersOtherMeans + \
    transformed_data_df.CommutersWorkingFromHome
    
transformed_data_df['CommutersApproxCars'] = transformed_data_df.CommutersDriving + \
    transformed_data_df.CommutersCarpooling/2

In [45]:
def create_commuter_fields_by_sex(data_df, sex):
    demographic_df = data_df[['Year', 'ZIPCode'] + \
                             [col for col in data_df.columns if sex in col and 'Commuters' in col]].copy()

    demographic_df[f'{sex}CommutersNonCar'] = demographic_df[f'{sex}CommutersPublicTransit'] + \
        demographic_df[f'{sex}CommutersBiking'] + demographic_df[f'{sex}CommutersTaxiMotorcycleAndOther'] + \
        demographic_df[f'{sex}CommutersWalking'] + + demographic_df[f'{sex}CommutersWorkingFromHome']

    demographic_df[f'{sex}CommutersTotal'] = demographic_df[f'{sex}CommutersDriving'] + \
        demographic_df[f'{sex}CommutersCarpooling'] + demographic_df[f'{sex}CommutersNonCar']
    
    demographic_df[f'{sex}CommutersApproxCars'] = demographic_df[f'{sex}CommutersDriving'] + \
        demographic_df[f'{sex}CommutersCarpooling']/2
    
    return(demographic_df)

def create_commuter_fields_by_demographic(data_df, demographic):
    demographic_df = data_df[['Year', 'ZIPCode'] + \
                             [col for col in data_df.columns 
                              if col[0:len(demographic)] == demographic and 'Commuters' in col]].copy()
    
    demographic_df[f'{demographic}CommutersNonCar'] = demographic_df[f'{demographic}CommutersPublicTransit'] + \
        demographic_df[f'{demographic}CommutersWalking'] + demographic_df[f'{demographic}CommutersTaxiBikeAndOther'] + \
        demographic_df[f'{demographic}CommutersWorkingFromHome']
    
    demographic_df[f'{demographic}CommutersApproxCars'] = demographic_df[f'{demographic}CommutersDriving'] + \
        demographic_df[f'{demographic}CommutersCarpooling']/2
    
    return(demographic_df)

def create_complete_occupations_fields_by_demographic(data_df, demographic):
    demographic_df = data_df[['Year', 'ZIPCode'] + \
                             [col for col in data_df.columns 
                              if col[0:len(demographic)] == demographic and 'Workers' in col]].copy()
    
    demographic_df[f'{demographic}Workers'] = demographic_df[f'{demographic}MaleWorkers'] + \
        demographic_df[f'{demographic}FemaleWorkers']
    demographic_df[f'{demographic}WorkersManagementBusinesScienceAndArts'] = demographic_df[f'{demographic}MaleWorkersManagementBusinesScienceAndArts'] + \
        demographic_df[f'{demographic}FemaleWorkersManagementBusinesScienceAndArts']
    demographic_df[f'{demographic}WorkersService'] = demographic_df[f'{demographic}MaleWorkersService'] + \
        demographic_df[f'{demographic}FemaleWorkersService']
    demographic_df[f'{demographic}WorkersSalesAndOffice'] = demographic_df[f'{demographic}MaleWorkersSalesAndOffice'] + \
        demographic_df[f'{demographic}FemaleWorkersSalesAndOffice']
    demographic_df[f'{demographic}WorkersResourcesConstructionAndMaintenance'] = demographic_df[f'{demographic}MaleWorkersResourcesConstructionAndMaintenance'] + \
        demographic_df[f'{demographic}FemaleWorkersResourcesConstructionAndMaintenance']
    demographic_df[f'{demographic}WorkersProductionTransportAndMaterialMoving'] = demographic_df[f'{demographic}MaleWorkersProductionTransportAndMaterialMoving'] + \
        demographic_df[f'{demographic}FemaleWorkersProductionTransportAndMaterialMoving']
    
    return(demographic_df)

In [46]:
general_demographics = transformed_data_df[[col for col in transformed_data_df.columns 
                                           if 'Commuters' not in col and 'Workers' not in col]]
commuters_by_sex = transformed_data_df[['Year', 'ZIPCode'] + [col for col in transformed_data_df.columns
                                       if col[0:9] == 'Commuters']]\
    .merge(create_commuter_fields_by_sex(transformed_data_df, 'Male'), on=['Year', 'ZIPCode'])\
    .merge(create_commuter_fields_by_sex(transformed_data_df, 'Female'),on=['Year', 'ZIPCode'])

commuters_by_age = create_commuter_fields_by_demographic(transformed_data_df, 'Age16to19')\
    .merge(create_commuter_fields_by_demographic(transformed_data_df, 'Age20to24'), on=['Year', 'ZIPCode'])\
    .merge(create_commuter_fields_by_demographic(transformed_data_df, 'Age25to44'), on=['Year', 'ZIPCode'])\
    .merge(create_commuter_fields_by_demographic(transformed_data_df, 'Age45to54'), on=['Year', 'ZIPCode'])\
    .merge(create_commuter_fields_by_demographic(transformed_data_df, 'Age55to59'), on=['Year', 'ZIPCode'])\
    .merge(create_commuter_fields_by_demographic(transformed_data_df, 'Age60to64'), on=['Year', 'ZIPCode'])\
    .merge(create_commuter_fields_by_demographic(transformed_data_df, 'Age65AndOver'), on=['Year', 'ZIPCode'])

commuters_by_race_ethnicity = create_commuter_fields_by_demographic(transformed_data_df, 'Black')\
    .merge(create_commuter_fields_by_demographic(transformed_data_df, 'Indigenous'), on=['Year', 'ZIPCode'])\
    .merge(create_commuter_fields_by_demographic(transformed_data_df, 'Asian'), on=['Year', 'ZIPCode'])\
    .merge(create_commuter_fields_by_demographic(transformed_data_df, 'PacificIslander'), on=['Year', 'ZIPCode'])\
    .merge(create_commuter_fields_by_demographic(transformed_data_df, 'OtherRace'), on=['Year', 'ZIPCode'])\
    .merge(create_commuter_fields_by_demographic(transformed_data_df, 'TwoOrMoreRaces'), on=['Year', 'ZIPCode'])\
    .merge(create_commuter_fields_by_demographic(transformed_data_df, 'White'), on=['Year', 'ZIPCode'])\
    .merge(create_commuter_fields_by_demographic(transformed_data_df, 'Hispanic'), on=['Year', 'ZIPCode'])

In [47]:
worker_occupations_cols = [col for col in transformed_data_df.columns.tolist() if 'Worker' in col]

occupations_by_race_ethnicity_df = transformed_data_df[['Year', 'ZIPCode'] + [
    col for col in worker_occupations_cols if 'Black' in col or 'Indigenous' in col or 'Asian' in col or
    'PacificIslander' in col or 'OtherRace' in col or 'TwoOrMoreRaces' in col or 'Hispanic' in col
]]

occupations_overall_df = transformed_data_df[['Year', 'ZIPCode'] + [
    col for col in worker_occupations_cols if col not in occupations_by_race_ethnicity_df.columns
]]

In [48]:
occupations_by_sex_and_race_df = create_complete_occupations_fields_by_demographic(transformed_data_df, 'Black')\
    .merge(create_complete_occupations_fields_by_demographic(transformed_data_df, 'Indigenous'), on=['Year', 'ZIPCode'])\
    .merge(create_complete_occupations_fields_by_demographic(transformed_data_df, 'Asian'), on=['Year', 'ZIPCode'])\
    .merge(create_complete_occupations_fields_by_demographic(transformed_data_df, 'PacificIslander'), on=['Year', 'ZIPCode'])\
    .merge(create_complete_occupations_fields_by_demographic(transformed_data_df, 'OtherRace'), on=['Year', 'ZIPCode'])\
    .merge(create_complete_occupations_fields_by_demographic(transformed_data_df, 'TwoOrMoreRaces'), on=['Year', 'ZIPCode'])\
    .merge(create_complete_occupations_fields_by_demographic(transformed_data_df, 'White'), on=['Year', 'ZIPCode'])\
    .merge(create_complete_occupations_fields_by_demographic(transformed_data_df, 'Hispanic'), on=['Year', 'ZIPCode'])

occupations_by_race_ethnicity_df = occupations_by_sex_and_race_df[[
    col for col in occupations_by_sex_and_race_df.columns.tolist() if 'MaleWorkers' not in col 
    and 'FemaleWorkers' not in col
]]

In [49]:
general_demographics.to_csv('Cleaned Data/LAGeneralDemographicsByZIPCode.csv', index=False)
commuters_by_sex.to_csv('Cleaned Data/LACommutingDataGeneralAndSexByZIPCode.csv', index=False)
commuters_by_age.to_csv('Cleaned Data/LACommutingDataAgeByZIPCode.csv', index=False)
commuters_by_race_ethnicity.to_csv('Cleaned Data/LACommutingDataRaceEthnicityByZIPCode.csv', index=False)
occupations_overall_df.to_csv('Cleaned Data/LAWorkerOccupationsOverallByZIPCode.csv', index=False)
occupations_by_race_ethnicity_df.to_csv('Cleaned Data/LAWorkerOccupationsRaceEthnicityByZIPCode.csv', index=False)
occupations_by_sex_and_race_df.to_csv('Cleaned Data/LAWorkerOccupationsSexRaceEthnicityByZIPCode.csv', index=False)

## Creating a field name translation table to make stacking the data easier

In [50]:
general_demographics.head()

,Year,ZIPCode,HispanicPopulation,IndigenousNonHispanicPopulation,AsianNonHispanicPopulation,BlackNonHispanicPopulation,PacificIslanderNonHispanicPopulation,OtherRaceNonHispanicPopulation,TwoOrMoreRacesNonHispanicPopulation,WhiteNonHispanicPopulation,IndigenousPopulation,AsianPopulation,BlackPopulation,PacificIslanderPopulation,OtherRacePopulation,WhitePopulation,TwoOrMoreRacesPopulation,Age10to14,Age15to19,Age20to24,Age25to34,Age35to44,Age45to54,Age5to9,Age55to59,Age60to64,Age65to74,Age75to84,Age85AndOver,FemalePopulation,MalePopulation,AgeUnder5
0,2011,90001,49339,19,48,4819,0,14,56,465,309,59,4943,0,8609,40379,461,4935,5363,5078,9308,7668,6161,4995,1894,1677,1579,735,188,27764,26996,5179
1,2011,90002,33232,20,65,12716,30,10,91,345,70,65,12844,30,5029,28035,436,5144,4733,4243,6596,6047,5472,4363,1518,996,1567,868,350,24038,22471,4612
2,2011,90003,48261,236,115,16430,43,51,282,765,480,115,16596,43,16372,31870,707,5989,6608,5921,9964,8621,7287,6378,2239,1939,2053,1227,460,34166,32017,7497
3,2011,90004,32551,76,17154,1973,182,481,874,10641,322,17309,2075,182,19692,23007,1345,3560,4016,5379,11777,10852,9167,3321,3685,2700,3546,1338,574,31536,32396,4017
4,2011,90005,22426,109,12739,2314,389,126,150,2925,342,12845,2625,389,14354,10152,471,2311,2201,3182,8829,7157,5770,2205,1851,1225,1902,1333,290,19620,21558,2922


In [51]:
all_field_names = list(general_demographics.columns.tolist()[2:] + commuters_by_sex.columns.tolist()[2:] + 
                       commuters_by_age.columns.tolist()[2:] + commuters_by_race_ethnicity.columns.tolist()[2:] + 
                       occupations_by_race_ethnicity_df.columns.tolist()[2:] + occupations_overall_df.columns.tolist()[2:] +
                       occupations_by_sex_and_race_df.columns.tolist()[2:])

all_field_names

['HispanicPopulation',
 'IndigenousNonHispanicPopulation',
 'AsianNonHispanicPopulation',
 'BlackNonHispanicPopulation',
 'PacificIslanderNonHispanicPopulation',
 'OtherRaceNonHispanicPopulation',
 'TwoOrMoreRacesNonHispanicPopulation',
 'WhiteNonHispanicPopulation',
 'IndigenousPopulation',
 'AsianPopulation',
 'BlackPopulation',
 'PacificIslanderPopulation',
 'OtherRacePopulation',
 'WhitePopulation',
 'TwoOrMoreRacesPopulation',
 'Age10to14',
 'Age15to19',
 'Age20to24',
 'Age25to34',
 'Age35to44',
 'Age45to54',
 'Age5to9',
 'Age55to59',
 'Age60to64',
 'Age65to74',
 'Age75to84',
 'Age85AndOver',
 'FemalePopulation',
 'MalePopulation',
 'AgeUnder5',
 'CommutersBiking',
 'CommutersCarpooling',
 'CommutersDriving',
 'CommutersMotorcycle',
 'CommutersOtherMeans',
 'CommutersPublicTransit',
 'CommutersTaxi',
 'CommutersWalking',
 'CommutersWorkingFromHome',
 'CommutersNonCar',
 'CommutersApproxCars',
 'MaleCommutersBiking',
 'MaleCommutersCarpooling',
 'MaleCommutersDriving',
 'MaleCommut

In [52]:
field_groupings = {
    'Total Population': 'Total Population',
    'Male': 'Sex', 
    'Female': 'Sex',
    'Black': 'Race', 
    'Indigenous': 'Race', 
    'Asian': 'Race', 
    'Pacific Islander': 'Race', 
    'Other Race': 'Race', 
    'Two or More Races': 'Race', 
    'White': 'Race', 
    'Black Non-Hispanic': 'Race or Ethnicity', 
    'Indigenous Non-Hispanic': 'Race or Ethnicity', 
    'Asian Non-Hispanic': 'Race or Ethnicity', 
    'Pacific Islander Non-Hispanic': 'Race or Ethnicity', 
    'Other Race Non-Hispanic': 'Race or Ethnicity', 
    'Two or More Races Non-Hispanic': 'Race or Ethnicity', 
    'White Non-Hispanic': 'Race or Ethnicity', 
    'Hispanic': 'Race or Ethnicity'
}

def id_demographic(input_field_name):
    if 'Commuters' in input_field_name:
        field_name = input_field_name.replace('Population', '').split('Commuters')[0]
    elif 'Workers' in input_field_name:
        field_name = input_field_name.replace('Population', '').split('Workers')[0]
    else:
        field_name = input_field_name
    if field_name == '':
        demographic_name = 'Total Population'
    elif field_name[:6] == 'Female':
        demographic_name = 'Female'
    elif field_name[:4] == 'Male':
        demographic_name = 'Male'
    elif field_name[:3] == 'Age':
        if 'AndOver' in field_name:
            age_split = field_name[3:].split('AndOver')
            demographic_name = age_split[0] + ' Years and Over'
        elif 'Under' in field_name:
            age_split = field_name[3:].split('Under')
            demographic_name = 'Under ' + age_split[1] + ' Years'
        else:
            age_split = field_name[3:].split('to')
            demographic_name = age_split[0] + ' to ' + age_split[1] + ' Years'
    else:
        demographic_name = ' '.join(re.findall('[A-Z][^A-Z]*', field_name))\
            .replace(' Or ', ' or ')\
            .replace('Non Hispanic', 'Non-Hispanic')
    if 'Population' in demographic_name and demographic_name != 'Total Population':
        demographic_name = demographic_name.split('Population')[0]
    return(demographic_name.strip())


    
translated_field_names_df = pd.DataFrame({'FieldName': all_field_names}).drop_duplicates()
translated_field_names_df['Demographic'] = [id_demographic(field_name) 
                                            for field_name in translated_field_names_df.FieldName.values]
translated_field_names_df['DemographicType'] = [field_groupings[demographic] if demographic in field_groupings.keys()
                                                else field_groupings[demographic.split('Male')[0]]  
                                                    if demographic.split('Male')[0] in field_groupings.keys() 
                                                else field_groupings[demographic.split('Female')[0]]  
                                                    if demographic.split('Female')[0] in field_groupings.keys()
                                                else 'Age' if ' Years' in demographic else None
                                                for demographic in translated_field_names_df.Demographic.values]
translated_field_names_df['CommuterType'] = [' '.join(re.findall('[A-Z][^A-Z]*', field_name.split('Commuters')[1]))\
                                                 .replace(' And ',  ' and ').replace(' From ', ' from ')\
                                                 .replace(' Motorcycle ', ', Motorcycle, ')\
                                                 .replace(' Bike ', ', Bike, ')\
                                                 .replace('Non Car', 'Non-Car')\
                                                 .replace('Approx Cars', 'Approximate Number of Cars')
                                            if 'Commuters' in field_name else None for field_name 
                                             in translated_field_names_df.FieldName.values]
translated_field_names_df['CommuterTypeGroup'] = ['Taxi, Bike, and Other' if commuter_type in
                                                 ['Taxi', 'Biking', 'Motorcycle', 'OtherMeans', 
                                                  'Taxi, Bike, and Other', 'Taxi, Motorcycle, and Other']
                                                 else commuter_type 
                                                  if commuter_type else None for commuter_type 
                                                  in translated_field_names_df.CommuterType.values]
translated_field_names_df['WorkerType'] = [', '.join(re.findall('[A-Z][^A-Z]*', field_name.split('Workers')[1]))\
                                               .replace(', And, ', ', and ')\
                                               .replace('Transport,', 'Transportation,')\
                                               .replace('Material, Moving', 'Material Moving')\
                                               .replace('Sales, and', 'Sales and')
                                          if 'Workers' in field_name and 'Industry' not in field_name 
                                           and 'Class' not in field_name and field_name[-7:] != 'Workers'
                                           else None for field_name in translated_field_names_df.FieldName.values]
translated_field_names_df['WorkerIndustryType'] = [', '.join(re.findall('[A-Z][^A-Z]*', 
                                                                  field_name.split('Workers')[1].split('Industry')[0]))\
                                                 .replace(', And, ', ', and ')\
                                                 .replace('Hunting, Mining', 'Hunting, and Mining')\
                                                 .replace('Food, Services', 'Food Services')\
                                                 .replace('Real, Estate', 'Real Estate')\
                                                 .replace('Other, Services', 'Other Services')\
                                                 .replace('Waste, Management', 'Waste Management')\
                                                 .replace('Social, Assistance', 'Social Assistance')\
                                                 .replace('Public, Administration', 'Public Administration')\
                                                 .replace('Retail, Trade', 'Retail Trade')\
                                                 .replace('Wholesale, Trade', 'Wholesale Trade')
                                            
                                            if 'Workers' in field_name and 'Industry' in field_name 
                                            else None for field_name in translated_field_names_df.FieldName.values]
translated_field_names_df['WorkerClassType'] = [' '.join(re.findall('[A-Z][^A-Z]*', 
                                                                  field_name.split('Workers')[1].split('Class')[0]))\
                                                 .replace(' And ', ' and ')\
                                                 .replace('Self Employed', 'Self-Employed')\
                                                 .replace('Not Incorporated', 'Not-Incorporated')
                                            
                                            if 'Workers' in field_name and 'Class' in field_name 
                                            else None for field_name in translated_field_names_df.FieldName.values]

In [53]:
translated_field_names_df

,FieldName,Demographic,DemographicType,CommuterType,CommuterTypeGroup,WorkerType,WorkerIndustryType,WorkerClassType
0,HispanicPopulation,Hispanic,Race or Ethnicity,None,None,None,None,None
1,IndigenousNonHispanicPopulation,Indigenous Non-Hispanic,Race or Ethnicity,None,None,None,None,None
2,AsianNonHispanicPopulation,Asian Non-Hispanic,Race or Ethnicity,None,None,None,None,None
3,BlackNonHispanicPopulation,Black Non-Hispanic,Race or Ethnicity,None,None,None,None,None
4,PacificIslanderNonHispanicPopulation,Pacific Islander Non-Hispanic,Race or Ethnicity,None,None,None,None,None
5,OtherRaceNonHispanicPopulation,Other Race Non-Hispanic,Race or Ethnicity,None,None,None,None,None
6,TwoOrMoreRacesNonHispanicPopulation,Two or More Races Non-Hispanic,Race or Ethnicity,None,None,None,None,None
7,WhiteNonHispanicPopulation,White Non-Hispanic,Race or Ethnicity,None,None,None,None,None
8,IndigenousPopulation,Indigenous,Race,None,None,None,None,None
9,AsianPopulation,Asian,Race,None,None,None,None,None


In [54]:
translated_field_names_df.to_csv('Cleaned Data/APIDataFieldNameTranslations.csv', index=False)